### Links and Resources
| Concept / Command | Description |
|---|---|
| [Delta as a streaming source](https://docs.delta.io/latest/delta-streaming.html#delta-table-as-a-source) | Reading a Delta table as a stream |
| [Delta as a streaming sink](https://docs.delta.io/latest/delta-streaming.html#delta-table-as-a-sink) | Writing a stream to a Delta table |
| [Change Data Feed](https://docs.databricks.com/en/delta/delta-change-data-feed.html) | Streaming row-level changes (insert / update / delete) |
| [`startingVersion`](https://docs.delta.io/latest/delta-streaming.html#specify-initial-position) | Start stream from a specific Delta version |
| [`maxFilesPerTrigger`](https://docs.delta.io/latest/delta-streaming.html#limit-input-rate) | Rate-limit the number of Delta files per micro-batch |

In [ ]:
import os, sys

try:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.getOrCreate()
except ImportError:
    if not os.getenv("DATABRICKS_RUNTIME_VERSION"):
        raise RuntimeError(
            f"Wrong Python kernel — 'databricks.connect' not found in:\n  {sys.executable}\n\n"
            "In VSCode: click the kernel selector (top-right of notebook) "
            "and choose the interpreter at:\n"
            r"  C:\Source\Dsa\DataBricks\.venv\Scripts\python.exe"
        )
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.getOrCreate()


def display(df, n=20):
    df.show(n, truncate=False)

# Streaming with Delta Lake

Delta Lake is the natural partner for Structured Streaming on Databricks. It works as both **source** and **sink**, and it adds several capabilities that neither plain files nor Kafka can provide:

| Feature | What it gives you |
|---|---|
| **ACID source reads** | The stream always reads a consistent snapshot — no partial writes |
| **Schema enforcement** | The sink rejects rows that don't match the table schema |
| **Change Data Feed (CDF)** | Stream the row-level changelog: inserts, updates (pre/post), deletes |
| **`startingVersion`** | Start reading from any historical Delta version — clean backfills |
| **`Trigger.AvailableNow`** | Process all new Delta commits since last checkpoint, then stop |
| **Exactly-once writes** | Delta's transaction log makes sink writes idempotent — safe to replay |

## 1. Delta table as a streaming source

Reading a Delta table as a stream is identical to reading a batch table — just swap `spark.read` for `spark.readStream`. Every time a new transaction commits to the source table, the stream picks it up as a new micro-batch.

In [ ]:
from pyspark.sql import functions as F
import time, uuid

CKPT_BASE = "/Volumes/population_metrics/landing/datasets/streaming_demo/_checkpoints"

# ── Create and seed a source Delta table ─────────────────────────────────────
spark.sql(
    "CREATE OR REPLACE TABLE population_metrics.default.delta_src "
    "(event_id LONG, user_id INT, amount INT, created_at TIMESTAMP) USING DELTA"
)

spark.sql(
    "INSERT INTO population_metrics.default.delta_src VALUES "
    "(1, 0, 50,  current_timestamp()), "
    "(2, 1, 30,  current_timestamp()), "
    "(3, 2, 80,  current_timestamp())"
)

print("Initial rows:")
spark.sql("SELECT * FROM population_metrics.default.delta_src").show()

# ── Open as a streaming source ────────────────────────────────────────────────
delta_stream = (spark.readStream
    .format("delta")
    .table("population_metrics.default.delta_src"))

print("isStreaming:", delta_stream.isStreaming)

# First pass: read all current rows then stop.
# Under availableNow we only see rows that exist at start; for the "late arrival" demo
# we insert new rows and run a second availableNow query against the same checkpoint.
CKPT_DELTA_SRC = f"{CKPT_BASE}/delta_src_demo_{uuid.uuid4().hex[:8]}"
q = (delta_stream.writeStream
     .format("memory").queryName("delta_src_demo")
     .outputMode("append")
     .option("checkpointLocation", CKPT_DELTA_SRC)
     .trigger(availableNow=True)
     .start())
q.awaitTermination()
print(f"\nRows seen in first pass: {q.lastProgress['numInputRows']} in batch {q.lastProgress['batchId']}")

# ── Append new data — second pass picks up only the new rows (checkpoint resume) ──
spark.sql(
    "INSERT INTO population_metrics.default.delta_src VALUES "
    "(4, 3, 45,  current_timestamp()), "
    "(5, 4, 70,  current_timestamp())"
)

q2 = (delta_stream.writeStream
      .format("memory").queryName("delta_src_demo")
      .outputMode("append")
      .option("checkpointLocation", CKPT_DELTA_SRC)   # same checkpoint resumes from prior offset
      .trigger(availableNow=True)
      .start())
q2.awaitTermination()

print("\nAll rows seen across both passes:")
spark.sql("SELECT * FROM delta_src_demo ORDER BY event_id").show()

## 2. Source options — controlling what and how much to read

| Option | Default | Purpose |
|---|---|---|
| `startingVersion` | latest | Start from a specific Delta version (0 = all history) |
| `startingTimestamp` | latest | Start from the version that was current at a given timestamp |
| `maxFilesPerTrigger` | 1000 | Max Parquet files per micro-batch — rate-limits large tables |
| `maxBytesPerTrigger` | unlimited | Max bytes per micro-batch — alternative to file-count rate limiting |
| `ignoreDeletes` | false | Allow source table to have rows deleted without failing the stream |
| `ignoreChanges` | false | Allow updates/deletes (needed when CDF is off) — may see duplicates |

In [ ]:
# startingVersion=0 — read the full Delta history from version 0 (all rows ever written)
stream_from_v0 = (spark.readStream
    .format("delta")
    .option("startingVersion", 0)
    .table("population_metrics.default.delta_src"))

# maxFilesPerTrigger — useful when a large Delta table has thousands of files;
# prevents one giant batch from overwhelming the cluster
stream_rate_limited = (spark.readStream
    .format("delta")
    .option("startingVersion", 0)
    .option("maxFilesPerTrigger", 1)    # one Parquet file per micro-batch
    .table("population_metrics.default.delta_src"))

q = (stream_from_v0.writeStream
     .format("memory").queryName("from_v0")
     .outputMode("append")
     .option("checkpointLocation", f"{CKPT_BASE}/from_v0_{uuid.uuid4().hex[:8]}")
     .trigger(availableNow=True).start())
q.awaitTermination(20)

print(f"startingVersion=0: {len(q.recentProgress)} batch(es), total input rows across all batches:")
total = sum(p["numInputRows"] for p in q.recentProgress)
print(f"  {total} rows (all versions of the source table)")

## 3. Delta table as a streaming sink

Writing a stream to Delta is the most common streaming pattern on Databricks. Delta's transaction log makes each micro-batch write atomic and idempotent — if the query restarts mid-batch, the partial write is rolled back and replayed cleanly.

In [ ]:
CKPT_SINK = f"{CKPT_BASE}/delta_sink_demo_{uuid.uuid4().hex[:8]}"

# Stream from rate-micro-batch source → transform → Delta sink (availableNow on serverless)
q = (spark.readStream.format("rate-micro-batch").option("rowsPerBatch", 25).load()
    .withColumnRenamed("timestamp", "event_time")
    .withColumnRenamed("value",     "event_id")
    .withColumn("user_id", (F.col("event_id") % 5).cast("int"))
    .withColumn("amount",  (F.rand() * 100 + 1).cast("int"))
    .select("event_id", "user_id", "amount", "event_time")
    .writeStream
    .format("delta")
    .option("checkpointLocation", CKPT_SINK)
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable("population_metrics.default.delta_sink_demo")
)
q.awaitTermination()

row_count = spark.sql("SELECT COUNT(*) FROM population_metrics.default.delta_sink_demo").collect()[0][0]
print(f"Rows in Delta sink: {row_count}")
print("\nDelta table history (each entry is one micro-batch):")
spark.sql("DESCRIBE HISTORY population_metrics.default.delta_sink_demo").select(
    "version", "timestamp", "operation", "operationParameters"
).show(5, truncate=60)

## 4. Change Data Feed (CDF)

**Change Data Feed** exposes every row-level change committed to a Delta table as a structured stream of CDC events.

Each CDF event has three extra columns:

| Column | Values | Meaning |
|---|---|---|
| `_change_type` | `insert` | A new row was added |
| | `update_preimage` | The row value **before** an update |
| | `update_postimage` | The row value **after** an update |
| | `delete` | A row was deleted |
| `_commit_version` | long | The Delta version this change belongs to |
| `_commit_timestamp` | timestamp | When the change was committed |

**CDF must be enabled on the table** before changes start being recorded:

### Enabling CDF

```sql
-- At table creation:
CREATE TABLE my_table (...) TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')

-- On an existing table:
ALTER TABLE my_table SET TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')
```

CDF only records changes **after** it is enabled. Past history is not retroactively captured.

In [ ]:
# ── Create a CDF-enabled table ───────────────────────────────────────────────
spark.sql(
    "CREATE OR REPLACE TABLE population_metrics.default.cdf_demo "
    "(id INT, name STRING, score INT) "
    "TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')"
)

# ── Record some changes ───────────────────────────────────────────────────────
spark.sql("INSERT INTO population_metrics.default.cdf_demo VALUES (1,'alice',80),(2,'bob',60),(3,'carol',90)")
spark.sql("UPDATE population_metrics.default.cdf_demo SET score = 95 WHERE name = 'alice'")
spark.sql("DELETE FROM population_metrics.default.cdf_demo WHERE name = 'bob'")
spark.sql("INSERT INTO population_metrics.default.cdf_demo VALUES (4,'dave',75)")

print("Current table state:")
spark.sql("SELECT * FROM population_metrics.default.cdf_demo ORDER BY id").show()

print("Delta history:")
spark.sql("DESCRIBE HISTORY population_metrics.default.cdf_demo").select(
    "version","operation"
).show(10, truncate=False)

In [ ]:
CKPT_CDF = f"{CKPT_BASE}/cdf_events_{uuid.uuid4().hex[:8]}"

# ── Read CDF as a stream, starting from version 0 ────────────────────────────
cdf_stream = (spark.readStream
    .format("delta")
    .option("readChangeFeed", "true")
    .option("startingVersion", 0)
    .table("population_metrics.default.cdf_demo"))

q = (cdf_stream.writeStream
     .format("memory").queryName("cdf_events")
     .outputMode("append")
     .option("checkpointLocation", CKPT_CDF)
     .trigger(availableNow=True)
     .start())

q.awaitTermination(30)

print("All CDC events (insert / update_preimage / update_postimage / delete):")
spark.sql(
    "SELECT _change_type, _commit_version, id, name, score "
    "FROM cdf_events "
    "ORDER BY _commit_version, _change_type, id"
).show(truncate=False)

## 5. A full Delta-to-Delta streaming pipeline

The most common production pattern on Databricks: read raw events from a Bronze Delta table, clean and transform them, write to a Silver Delta table — both streaming continuously with a checkpoint.

In [ ]:
CKPT_PIPELINE = f"{CKPT_BASE}/pipeline_{uuid.uuid4().hex[:8]}"

# Bronze → Silver pipeline: filter bad records, parse, write cleaned data
silver_stream = (
    spark.readStream
        .format("delta")
        .option("startingVersion", 0)
        .table("population_metrics.default.delta_src")
    # filter: drop nulls and negative amounts
    .filter(F.col("amount") > 0)
    .filter(F.col("user_id").isNotNull())
    # enrich
    .withColumn("amount_bucket",
        F.when(F.col("amount") < 50, "low")
         .when(F.col("amount") < 80, "mid")
         .otherwise("high"))
    .withColumn("processed_at", F.current_timestamp())
)

q = (silver_stream.writeStream
     .format("delta")
     .option("checkpointLocation", CKPT_PIPELINE)
     .outputMode("append")
     .trigger(availableNow=True)
     .toTable("population_metrics.default.delta_silver_demo"))

q.awaitTermination()

print("Silver table contents:")
spark.sql(
    "SELECT event_id, user_id, amount, amount_bucket, processed_at "
    "FROM population_metrics.default.delta_silver_demo ORDER BY event_id"
).show(truncate=False)

## 6. Schema enforcement on the sink

Delta enforces the table schema on every write. If a streaming batch produces a column with the wrong type or an unexpected column (without schema evolution enabled), the query fails with a clear error — rather than silently writing bad data.

```python
# This will raise AnalysisException if 'extra_col' is not in the target schema:
(stream.withColumn("extra_col", F.lit("oops"))
       .writeStream.toTable("existing_table"))

# To add new columns automatically, enable schema evolution (mergeSchema):
(stream.writeStream
       .option("mergeSchema", "true")
       .toTable("existing_table"))
```

Schema enforcement is one of the main reasons to prefer Delta over plain Parquet for streaming sinks — with Parquet you would silently write incompatible files and only discover the problem at read time.

## Takeaways

- Reading a Delta table as a stream with `spark.readStream.format("delta").table(...)` picks up new commits automatically — no polling code needed.
- `startingVersion=0` reads full history; `startingVersion="latest"` (default) reads only future changes.
- `maxFilesPerTrigger` rate-limits large Delta tables to prevent overwhelming the cluster on the first run.
- Delta sinks provide exactly-once writes via the transaction log — safe to replay after a crash.
- **Change Data Feed** exposes row-level `insert` / `update_preimage` / `update_postimage` / `delete` events as a stream. Enable it with `TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')`.
- Schema enforcement on the Delta sink catches data quality issues immediately — before they reach downstream consumers.